In [ ]:
# Install & Import Libraries

# Required libraries
import pandas as pd
import numpy as np
import tensorflow as tf
import shap
import datetime
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Conv1D, Dropout, AveragePooling1D, Flatten, Dense, concatenate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# Define Data and Model Paths

# Define input data and model paths
data_path = "/home/faiza/vienna/mmmm/30nt_CRISPRon_Scores data - Copy.xlsx"
model_path = "/home/faiza/vienna/mmmm/data/deep_models/best/1.model.best"


In [ ]:
# Load Data Function

# Load Excel dataset
def load_data(file_path):
    return pd.read_excel(file_path)


In [ ]:
# One-hot Encode the 30-nt Sequences

# One-hot encode 30-nt gRNA sequences
def one_hot_encode_sequences(sequences):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    encoded = np.zeros((len(sequences), 30, 4), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for j in range(30):
            encoded[i,j] = mapping.get(seq[j].upper(), [0,0,0,0])
    return encoded


In [ ]:
# Calculate DeltaGb Thermodynamic Feature

# Calculate DeltaGb based on GC content
def calculate_deltaGb(sequence):
    gc_content = (sequence.count('G') + sequence.count('C')) / 30
    return -15.3 + (gc_content - 0.5) * 8.2


In [ ]:
# Normalize DeltaGb and Prepare Feature Array

# Normalize and reshape DeltaGb feature
def process_features(data):
    data['DeltaGb'] = data['Sequence'].apply(calculate_deltaGb)
    return ((data['DeltaGb'] + 15.3) / 4.1).values.reshape(-1,1).astype(np.float32)


In [ ]:
# CrisprOn Model Architecture

# Build model architecture with 3 convolutional branches + DeltaGb input
def build_model():
    seq_input = Input(shape=(30,4), name='input_onehot')
    deltaGb_input = Input(shape=(1,), name='input_dGB')

    def conv_branch(x, fs, f):
        x = Conv1D(f, fs, activation='relu')(x)
        x = Dropout(0.3)(x)
        x = AveragePooling1D(2)(x)
        return Flatten()(x)
    
    merged = concatenate([
        conv_branch(seq_input, 3, 100),
        conv_branch(seq_input, 5, 70),
        conv_branch(seq_input, 7, 40),
        deltaGb_input
    ])
    
    x = Dense(80, activation='relu')(merged)
    x = Dropout(0.3)(x)
    x = Dense(80, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(60, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs=[seq_input, deltaGb_input], outputs=Dense(1)(x))


In [ ]:
# Extracting SHAP values on bases of original 121 features

# Wrapper for SHAP to make predictions in correct format
def model_predict(data):
    if isinstance(data, list):
        return model.predict(data)
    seq = data[:, :120].reshape(-1, 30, 4).astype(np.float32)
    deltaGb = data[:, 120:121].astype(np.float32)
    return model.predict([seq, deltaGb])


In [ ]:
# Load model and data

# Load data and model
df = load_data(data_path)

try:
    model = load_model(model_path)
    print("Model loaded successfully.")
except:
    model = build_model()
    model.load_weights(model_path)
    print("Built new model.")

model.compile(optimizer=tf.keras.optimizers.Adam(0.0001), loss='mse', metrics=['mae'])


In [ ]:
# Preprocess Input and Make Predictions

# Prepare input arrays and predict efficiency
seq_array = one_hot_encode_sequences(df['Sequence'])
deltaGb_array = process_features(df)
y_true = df['Scores CRISPRon'].values.astype(np.float32)

# Predict using model
df['CRISPRon_Prediction'] = model.predict([seq_array, deltaGb_array]).flatten()


In [ ]:
# Shap value calculation

# Prepare input arrays and predict efficiency
seq_array = one_hot_encode_sequences(df['Sequence'])
deltaGb_array = process_features(df)
y_true = df['Scores CRISPRon'].values.astype(np.float32)

# Predict using model
df['CRISPRon_Prediction'] = model.predict([seq_array, deltaGb_array]).flatten()


In [ ]:
# Save Predictions and SHAP Outputs

# Save results
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
df.to_excel(f'CRISPRon_Predictions_SHAP_{timestamp}.xlsx', index=False)
print(f"\nResults saved to: CRISPRon_Predictions_SHAP_{timestamp}.xlsx")



In [ ]:
# SHAP waterfall plot

# Create SHAP waterfall plot for first sample
plt.figure(figsize=(12,6))
f_x = df['CRISPRon_Prediction'].iloc[0]
E_x = df['CRISPRon_Prediction'].mean()

waterfall_data = shap.Explanation(
    values=shap_values[0].values,
    base_values=E_x,
    data=combined_input[0],
    feature_names=explainer.feature_names
)

shap.plots.waterfall(waterfall_data, max_display=20)
plt.title(f"First Sample SHAP Values\nf(x) = {f_x:.4f}, E(x) = {E_x:.4f}")
plt.savefig(f'SHAP_Waterfall_{timestamp}.png', bbox_inches='tight')
plt.close()


In [ ]:
# Finall code


import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Conv1D, Dropout, AveragePooling1D, Flatten, Dense, concatenate
import datetime
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shap
import matplotlib.pyplot as plt

# Define paths
data_path = "/home/faiza/vienna/mmmm/30nt_CRISPRon_Scores data - Copy.xlsx"
model_path = "/home/faiza/vienna/mmmm/data/deep_models/best/1.model.best"

def load_data(file_path):
    return pd.read_excel(file_path)

def one_hot_encode_sequences(sequences):
    mapping = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}
    encoded = np.zeros((len(sequences), 30, 4), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for j in range(30):
            encoded[i,j] = mapping.get(seq[j].upper(), [0,0,0,0])
    return encoded

def calculate_deltaGb(sequence):
    gc_content = (sequence.count('G') + sequence.count('C')) / 30
    return -15.3 + (gc_content - 0.5) * 8.2

def process_features(data):
    data['DeltaGb'] = data['Sequence'].apply(calculate_deltaGb)
    return ((data['DeltaGb'] + 15.3) / 4.1).values.reshape(-1,1).astype(np.float32)

def build_model():
    seq_input = Input(shape=(30,4), name='input_onehot')
    deltaGb_input = Input(shape=(1,), name='input_dGB')
    
    def conv_branch(x, fs, f):
        x = Conv1D(f, fs, activation='relu')(x)
        x = Dropout(0.3)(x)
        x = AveragePooling1D(2)(x)
        return Flatten()(x)
    
    merged = concatenate([
        conv_branch(seq_input, 3, 100),
        conv_branch(seq_input, 5, 70),
        conv_branch(seq_input, 7, 40),
        deltaGb_input
    ])
    
    x = Dense(80, activation='relu')(merged)
    x = Dropout(0.3)(x)
    x = Dense(80, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(60, activation='relu')(x)
    x = Dropout(0.3)(x)
    return Model(inputs=[seq_input, deltaGb_input], outputs=Dense(1)(x))

def model_predict(data):
    if isinstance(data, list):
        return model.predict(data)
    seq = data[:, :120].reshape(-1, 30, 4).astype(np.float32)
    deltaGb = data[:, 120:121].astype(np.float32)
    return model.predict([seq, deltaGb])

if __name__ == "__main__":
    # Load data and model
    df = load_data(data_path)
    try:
        model = load_model(model_path)
        print("Model loaded successfully.")
    except:
        model = build_model()
        model.load_weights(model_path)
        print("Built new model.")
    
    model.compile(optimizer=tf.keras.optimizers.Adam(0.0001), loss='mse', metrics=['mae'])
    
    # Prepare data
    seq_array = one_hot_encode_sequences(df['Sequence'])
    deltaGb_array = process_features(df)
    y_true = df['Scores CRISPRon'].values.astype(np.float32)
    
    # Predictions
    df['CRISPRon_Prediction'] = model.predict([seq_array, deltaGb_array]).flatten()
    
    # SHAP Analysis
    print("\nCalculating SHAP values...")
    combined_input = np.hstack([seq_array.reshape(len(df), 120), deltaGb_array])
    
    explainer = shap.Explainer(
        model_predict,
        masker=shap.maskers.Independent(combined_input[:10]),
        feature_names=[
            *[f"Pos{i+1}_{nuc}" for i in range(30) for nuc in 'ACGT'],
            "DeltaGb"
        ]
    )
    shap_values = explainer(combined_input, max_evals=5000)  # Increased evals for accuracy
    
    # Save results
    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    df.to_excel(f'CRISPRon_Predictions_SHAP_{timestamp}.xlsx', index=False)
    print(f"\nResults saved to: CRISPRon_Predictions_SHAP_{timestamp}.xlsx")
    
    # Visualization
    plt.figure(figsize=(12,6))
    f_x = df['CRISPRon_Prediction'].iloc[0]  # Ensure using the exact prediction for the first sample
    E_x = df['CRISPRon_Prediction'].mean()

    # Create waterfall plot data
    waterfall_data = shap.Explanation(
        values=shap_values[0].values,
        base_values=E_x,
        data=combined_input[0],
        feature_names=explainer.feature_names
    )

    # Plot and save
    shap.plots.waterfall(waterfall_data, max_display=20)
    plt.title(f"First Sample SHAP Values\nf(x) = {f_x:.4f}, E(x) = {E_x:.4f}")
    plt.savefig(f'SHAP_Waterfall_{timestamp}.png', bbox_inches='tight')
    plt.close()
    
    print("\nEvaluation Metrics:")
    print(f"MAE: {mean_absolute_error(y_true, df['CRISPRon_Prediction']):.4f}")
    print(f"MSE: {mean_squared_error(y_true, df['CRISPRon_Prediction']):.4f}")
    print(f"R²: {r2_score(y_true, df['CRISPRon_Prediction']):.4f}")



